<a href="https://colab.research.google.com/github/conto-inesatto/PneumoSense/blob/main/PneumoSense.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#This cell is for import the images from kaggle (we could also change the dataset)
import kagglehub
chest_xray_pneumonia_path = kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia')

print('Data source import complete.')

In [ ]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import keras
from keras.models import Sequential
from keras.layers import Dense, Conv2D , MaxPool2D , Flatten , Dropout , BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix
from keras.callbacks import ReduceLROnPlateau
import cv2
import os

In [ ]:
labels = ['PNEUMONIA', 'NORMAL']
img_size = 150
def get_training_data(data_dir):
    data = []
    for label in labels:
        path = os.path.join(data_dir, label)  #concatenates path components
        class_num = labels.index(label)
        for img in os.listdir(path):
            try:
                img_arr = cv2.imread(os.path.join(path, img), cv2.IMREAD_GRAYSCALE) #convert an image as an array and then transform it to grays
                resized_arr = cv2.resize(img_arr, (img_size, img_size)) # Reshaping images to the selected size
                data.append([resized_arr, class_num])
            except Exception as e:
                print(e)   #when an error occour in the try it goes in e
    return data

In [ ]:
train_ds = get_training_data(os.path.join(chest_xray_pneumonia_path, 'chest_xray', 'train'))
test_ds = get_training_data(os.path.join(chest_xray_pneumonia_path, 'chest_xray', 'test'))
val_ds = get_training_data(os.path.join(chest_xray_pneumonia_path, 'chest_xray', 'val'))

In [ ]:
l = []
for i in train_ds:
    if(i[1] == 0):
        l.append("Pneumonia") #add a pneumonia/normal data
    else:
        l.append("Normal")
sns.set_style('darkgrid')    #set the backgrownd
sns.countplot(l)             #show the count using bars

In [ ]:
plt.figure(figsize = (5,5))                 #create figure size
plt.imshow(train_ds[0][0], cmap='gray')        #trasform an array to an img and convert it to gray, trani is for specific thet we are using the first image of our list


plt.title(labels[train_ds[0][1]])              #is used to set the title of the current plot on top

plt.figure(figsize = (5,5))
plt.imshow(train_ds[-1][0], cmap='gray')
plt.title(labels[train_ds[-1][1]])

In [ ]:
x_train = []   #cell for training,validation of results and the testing of the AI
y_train = []

x_val = []
y_val = []

x_test = []
y_test = []

for feature, label in train_ds:
    x_train.append(feature)
    y_train.append(label)

for feature, label in test_ds:
    x_test.append(feature)
    y_test.append(label)

for feature, label in val_ds:
    x_val.append(feature)
    y_val.append(label)

In [ ]:
import random
from sklearn.model_selection import train_test_split

# Get original dataset lengths (These are now just for information)
original_train_len = len(train_ds)
original_val_len = len(val_ds) # This will still be 16, but we won't use it for x_val, y_val directly
original_test_len = len(test_ds)

# I've found a small dataset so i will give to validation some photos from training
TARGET_TOTAL_TRAIN_AND_VAL_SAMPLES = min(original_train_len, 5216) # Maximum samples to draw from original train_ds for (train+val)
VAL_SPLIT_RATIO = 0.2 # Use 20% of TARGET_TOTAL_TRAIN_AND_VAL_SAMPLES for validation

TARGET_TEST_SAMPLES = min(original_test_len, 624) # Maximum samples for the test set

# Combine features and labels from the original train_ds into a list of (feature, label) tuples
full_train_data_tuples = [(item[0], item[1]) for item in train_ds]

# Shuffle the combined list to ensure balanced class distribution when subsetting
random.shuffle(full_train_data_tuples)

# Extract features and labels from the shuffled pool
x_full_train_data = [item[0] for item in full_train_data_tuples]
y_full_train_labels = [item[1] for item in full_train_data_tuples]

# Take the desired number of samples for the combined training and validation pool
x_pool = x_full_train_data[:TARGET_TOTAL_TRAIN_AND_VAL_SAMPLES]
y_pool = y_full_train_labels[:TARGET_TOTAL_TRAIN_AND_VAL_SAMPLES]

# Split the 'pool' into training and validation sets using stratified sampling
x_train, x_val, y_train, y_val = train_test_split(
    x_pool, y_pool,
    test_size=VAL_SPLIT_RATIO,
    stratify=y_pool, # Essential for maintaining class balance
    random_state=42 # For reproducibility
)

# Combine features and labels from the original test_ds into a list of (feature, label) tuples
full_test_data_tuples = [(item[0], item[1]) for item in test_ds]

# Shuffle the test data as well
random.shuffle(full_test_data_tuples)

x_full_test_data = [item[0] for item in full_test_data_tuples]
y_full_test_labels = [item[1] for item in full_test_data_tuples]

# Take the desired number of samples for the test set
x_test = x_full_test_data[:TARGET_TEST_SAMPLES]
y_test = y_full_test_labels[:TARGET_TEST_SAMPLES]


print(f"Final Training set size: {len(x_train)} samples.")
print(f"Final Validation set size: {len(x_val)} samples.")
print(f"Final Test set size: {len(x_test)} samples.")

In [ ]:
from sklearn.utils import class_weight
import numpy as np

# Calculate the class wheight
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = dict(enumerate(class_weights))

print(f"Pesi delle classi calcolati: {class_weight_dict}")


In [ ]:
import tensorflow as tf

# Enable mixed precision training
# This policy will automatically cast most operations to float16,
# while keeping some numerically stable operations (like BatchNormalization) in float32.
policy = tf.keras.mixed_precision.Policy('mixed_float16')
tf.keras.mixed_precision.set_global_policy(policy)

print('Mixed precision enabled: %s' % tf.keras.mixed_precision.global_policy().name)

# Now, redefine your main model (the one named 'model') *after* setting the global policy.
# The example 'model_test' below is just to illustrate the effect on layer dtypes.

model_test = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(150, 150, 1)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

print("\n--- Model Layer Output dtypes with Mixed Precision ---")
for layer in model_test.layers:
    print(f"Layer: {layer.name}, Output Dtype: {layer.compute_dtype}")


In [ ]:
# Normalize the data
x_train = np.array(x_train) / 255
x_val = np.array(x_val) / 255
x_test = np.array(x_test) / 255

In [ ]:
# resize data for deep learning
x_train = x_train.reshape(-1, img_size, img_size, 1)
y_train = np.array(y_train)

x_val = x_val.reshape(-1, img_size, img_size, 1)
y_val = np.array(y_val)

x_test = x_test.reshape(-1, img_size, img_size, 1)
y_test = np.array(y_test)

In [ ]:
# With data augmentation to prevent overfitting and handling the imbalance in dataset

datagen = ImageDataGenerator(
        featurewise_center=False,  # set input mean to 0 over the dataset
        samplewise_center=False,  # set each sample mean to 0
        featurewise_std_normalization=False,  # divide inputs by std of the dataset
        samplewise_std_normalization=False,  # divide each input by its std
        zca_whitening=False,  # apply ZCA whitening
        rotation_range = 30,  # randomly rotate images in the range (degrees, 0 to 180)
        zoom_range = 0.2, # Randomly zoom image
        width_shift_range=0.10,  # randomly shift images horizontally (fraction of total width)
        height_shift_range=0.10,  # randomly shift images vertically (fraction of total height)
        horizontal_flip = True,  # randomly flip images
        vertical_flip=False)  # randomly flip images


datagen.fit(x_train)

In [ ]:
import tensorflow as tf
import numpy as np

BATCH_SIZE = 32 # Increased batch size for better stability and efficiency

# Convert numpy arrays to TensorFlow datasets
train_ds_raw = tf.data.Dataset.from_tensor_slices((x_train, y_train))
val_ds_raw = tf.data.Dataset.from_tensor_slices((x_val, y_val))
test_ds_raw = tf.data.Dataset.from_tensor_slices((x_test, y_test))

# Create an augmentation model using Keras preprocessing layers
# This ensures all desired augmentations from ImageDataGenerator are applied efficiently.

augmenter = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(factor=0.2, fill_mode='nearest', seed=42), # Increased rotation_range slightly
    tf.keras.layers.RandomZoom(height_factor=0.2, width_factor=0.2, fill_mode='nearest', seed=42), # Increased zoom_range
    tf.keras.layers.RandomTranslation(height_factor=0.2, width_factor=0.2, fill_mode='nearest', seed=42), # Increased shift ranges
    tf.keras.layers.RandomFlip("horizontal", seed=42), # horizontal_flip = True
    tf.keras.layers.RandomBrightness(factor=0.2, seed=42), # Add random brightness
    tf.keras.layers.RandomContrast(factor=0.2, seed=42) # Add random contrast
    # vertical_flip was False in ImageDataGenerator, so we don't add RandomFlip("vertical")
    ])

def process_train_image(image, label):
    image = tf.cast(image, tf.float32) # Ensure float32 for augmentation layers
    image = augmenter(image, training=True) # Apply augmentation, training=True for random ops
    return image, label

def process_val_test_image(image, label):
    # Validation and test datasets should only be cast to float32 and normalized, not augmented
    return tf.cast(image, tf.float32), label

# Apply augmentation to the training dataset
train_ds = train_ds_raw.map(process_train_image, num_parallel_calls=tf.data.AUTOTUNE)
#for try data 07/05 da provare durante info
# Apply only casting to validation and test datasets
val_ds = val_ds_raw.map(process_val_test_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds_raw.map(process_val_test_image, num_parallel_calls=tf.data.AUTOTUNE)


# Batch and prefetch datasets for performance
train_ds = train_ds.shuffle(buffer_size=1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("tf.data pipelines (train_ds, val_ds, test_ds) created with improved augmentation.")

In [ ]:
import tensorflow as tf


model = tf.keras.Sequential()
model.add(tf.keras.layers.Conv2D(32 , (3,3) , strides = 1 , padding = 'same' , activation = 'relu' , input_shape = (150,150,1)))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.MaxPool2D((2,2) , strides = 2 , padding = 'same'))
model.add(tf.keras.layers.Conv2D(64 , (3,3) , strides = 1 , padding = 'same' , activation = 'relu'))
# model.add(tf.keras.layers.Dropout(0.1)) # Temporarily removed dropout
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.MaxPool2D((2,2) , strides = 2 , padding = 'same'))
model.add(tf.keras.layers.Conv2D(64 , (3,3) , strides = 1 , padding = 'same' , activation = 'relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.MaxPool2D((2,2) , strides = 2 , padding = 'same'))
model.add(tf.keras.layers.Conv2D(128 , (3,3) , strides = 1 , padding = 'same' , activation = 'relu'))
# model.add(tf.keras.layers.Dropout(0.2)) # Temporarily removed dropout
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.MaxPool2D((2,2) , strides = 2 , padding = 'same'))
model.add(tf.keras.layers.Conv2D(256 , (3,3) , strides = 1 , padding = 'same' , activation = 'relu'))
# model.add(tf.keras.layers.Dropout(0.2)) # Temporarily removed dropout
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.MaxPool2D((2,2) , strides = 2 , padding = 'same'))
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(units = 128 , activation = 'relu'))
# model.add(tf.keras.layers.Dropout(0.2)) # Temporarily removed dropout
model.add(tf.keras.layers.Dense(units = 1 , activation = 'sigmoid'))


model.compile(optimizer = 'rmsprop' , loss = 'binary_crossentropy' , metrics = ['accuracy'])
model.summary()

In [ ]:
learning_rate_reduction = ReduceLROnPlateau(monitor='val_accuracy', patience = 5, verbose=1,factor=0.1, min_lr=0.00001)

In [ ]:
history = model.fit(datagen.flow(x_train,y_train, batch_size = 32) ,epochs = 12 , validation_data = datagen.flow(x_val, y_val) ,callbacks = [learning_rate_reduction])

In [ ]:
print("Loss of the model is - " , model.evaluate(test_ds)[0])
print("Accuracy of the model is - " , model.evaluate(test_ds)[1]*100 , "%")

In [ ]:
epochs = [i for i in range(len(history.history['accuracy']))]
fig , ax = plt.subplots(1,2)
train_acc = history.history['accuracy']
train_loss = history.history['loss']
val_acc = history.history['val_accuracy']
val_loss = history.history['val_loss']
fig.set_size_inches(20,10)

ax[0].plot(epochs , train_acc , 'go-' , label = 'Training Accuracy')
ax[0].plot(epochs , val_acc , 'ro-' , label = 'Validation Accuracy')
ax[0].set_title('Training & Validation Accuracy')
ax[0].legend()
ax[0].set_xlabel("Epochs")
ax[0].set_ylabel("Accuracy")

ax[1].plot(epochs , train_loss , 'g-o' , label = 'Training Loss')
ax[1].plot(epochs , val_loss , 'r-o' , label = 'Validation Loss')
ax[1].set_title('Testing Accuracy & Loss')
ax[1].legend()
ax[1].set_xlabel("Epochs")
ax[1].set_ylabel("Training & Validation Loss")
plt.show()

In [ ]:
predictions = (model.predict(x_test) > 0.5).astype("int32")
predictions = predictions.reshape(1,-1)[0]
predictions[:15]

In [ ]:
print(classification_report(y_test, predictions, labels=[0], target_names = ['Pneumonia (Class 0)']))

In [ ]:
cm = confusion_matrix(y_test,predictions, labels=[0, 1])
cm

In [ ]:
cm = pd.DataFrame(cm , index = ['0','1'] , columns = ['0','1'])

In [ ]:
plt.figure(figsize = (10,10))
sns.heatmap(cm,cmap= "Blues", linecolor = 'black' , linewidth = 1 , annot = True, fmt='',xticklabels = labels,yticklabels = labels)

In [ ]:
correct = np.nonzero(predictions == y_test)[0]
incorrect = np.nonzero(predictions != y_test)[0]

In [ ]:
i = 0
for c in correct[:6]:
    plt.subplot(3,2,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(x_test[c].reshape(150,150), cmap="gray", interpolation='none')
    plt.title("Predicted Class {},Actual Class {}".format(predictions[c], y_test[c]))
    plt.tight_layout()
    i += 1

In [ ]:
i = 0
for c in incorrect[:6]:
    plt.subplot(3,2,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(x_test[c].reshape(150,150), cmap="gray", interpolation='none')
    plt.title("Predicted Class {},   Actual Class {}".format(predictions[c], y_test[c]))
    plt.tight_layout()
    i += 1